# Notebook 01 — MoTrPAC Exercise Signatures

**Goal:** Extract rat exercise gene signatures for 3 tissues (gastrocnemius, liver,
white adipose) at the 8-week training timepoint, map to human orthologs via HCOP,
and export top-150 up/down gene sets for LINCS querying.

**Outputs:**
- `data/processed/motrpac_{tissue}_8w_up.txt` — top-150 upregulated human genes
- `data/processed/motrpac_{tissue}_8w_down.txt` — top-150 downregulated human genes
- `data/processed/motrpac_signatures.parquet` — full DE table with ortholog mapping

**Data source options (set flag below):**
1. `R_EXPORT` — TSV files written by `scripts/extract_motrpac.R`
2. `GCP_DOWNLOAD` — auto-download from public MoTrPAC GCP bucket
3. `DEMO` — synthetic data for pipeline testing (no real data required)

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
from src import signatures as sig

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# ── Configuration ────────────────────────────────────────────────────────────
# Choose data source: 'R_EXPORT', 'GCP_DOWNLOAD', or 'DEMO'
DATA_SOURCE = 'GCP_DOWNLOAD'

TISSUES      = list(sig.TISSUES.keys())   # gastrocnemius, liver, white_adipose
TIMEPOINT    = '8w'
TOP_N        = 150    # genes per direction per tissue
FDR_CUTOFF   = 0.05
LOGFC_CUTOFF = 0.0    # additional |logFC| threshold (0 = FDR only)

RAW_DIR      = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
EXTERNAL_DIR  = Path('../data/external')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Tissues: {TISSUES}')
print(f'Data source: {DATA_SOURCE}')

In [ ]:
# ── Demo data generator ───────────────────────────────────────────────────────
# Used when DATA_SOURCE == 'DEMO' or real data is unavailable.
# Synthetic DE table mimics the expected format with biologically plausible
# exercise-responsive genes seeded at the top.

import random

DEMO_EXERCISE_GENES = {
    'gastrocnemius': {
        'up':   ['Ppargc1a', 'Nr4a3', 'Myh7', 'Acta1', 'Tfam', 'Cpt1b',
                  'Acsl1', 'Pdk4', 'Ldhb', 'Ckmt2', 'Cox6a1', 'Vegfa',
                  'Fos', 'Jun', 'Egr1', 'Hspb1', 'Hspb8', 'Myc'],
        'down':  ['Mstn', 'Atrogin1', 'Murf1', 'Il6', 'Tnf', 'Cdkn2a',
                  'Foxo3', 'Klf15', 'Redd1', 'Ankrd1'],
    },
    'liver': {
        'up':   ['Ppargc1a', 'Hmgcs2', 'Acox1', 'Cpt1a', 'Hadha', 'Acadl',
                  'G6pc', 'Pck1', 'Cyp7a1', 'Fgf21', 'Hpgd'],
        'down':  ['Fasn', 'Acc1', 'Scd1', 'Srebf1', 'Il1b', 'Tnf',
                   'Mcp1', 'Serpine1', 'Hmgcr'],
    },
    'white_adipose': {
        'up':   ['Ppargc1a', 'Ucp1', 'Cidea', 'Prdm16', 'Adipoq', 'Lipe',
                  'Angptl4', 'Fgf21', 'Fabp4'],
        'down':  ['Leptin', 'Il6', 'Tnf', 'Mcp1', 'Serpine1', 'Fasn',
                   'Scd1', 'Atgl'],
    },
}

RNG = np.random.default_rng(42)

def make_demo_de(tissue: str, n_genes: int = 3000) -> pd.DataFrame:
    """Generate synthetic DE table for testing."""
    up_seeds   = DEMO_EXERCISE_GENES[tissue]['up']
    down_seeds = DEMO_EXERCISE_GENES[tissue]['down']

    symbols = (
        up_seeds + down_seeds +
        [f'Rn{i:05d}' for i in range(n_genes - len(up_seeds) - len(down_seeds))]
    )
    logFC = RNG.normal(0, 0.5, len(symbols))
    # Force seeds to have strong signal
    for i, g in enumerate(up_seeds):
        logFC[i] = RNG.uniform(0.8, 2.5)
    for i, g in enumerate(down_seeds):
        logFC[len(up_seeds) + i] = RNG.uniform(-2.5, -0.8)

    pvals = np.clip(RNG.beta(0.3, 5, len(symbols)), 1e-10, 1.0)
    pvals[:len(up_seeds) + len(down_seeds)] = RNG.uniform(1e-8, 0.001,
                                                           len(up_seeds) + len(down_seeds))

    from scipy.stats import false_discovery_control
    adj_pvals = false_discovery_control(pvals, method='bh')

    return pd.DataFrame({
        'gene_symbol': symbols,
        'ensembl_rat': [f'ENSRNOG{i:011d}' for i in range(len(symbols))],
        'logFC': logFC,
        'pvalue': pvals,
        'adj_pvalue': adj_pvals,
        'comparison_group': f'8w_trained_vs_control',
        'tissue': tissue,
    })

print('Demo data generator ready.')

In [ ]:
# ── Load MoTrPAC DE data ──────────────────────────────────────────────────────

de_tables = {}

for tissue in TISSUES:
    print(f'\n=== {tissue} ===')

    if DATA_SOURCE == 'DEMO':
        df = make_demo_de(tissue)
        print(f'  Demo: {len(df)} synthetic DE records')

    elif DATA_SOURCE == 'R_EXPORT':
        tsv = RAW_DIR / f'motrpac_{tissue}_{TIMEPOINT}_r_export.tsv'
        if not tsv.exists():
            print(f'  R export not found at {tsv}; falling back to DEMO')
            df = make_demo_de(tissue)
        else:
            df = sig.load_motrpac_from_r_export(tsv)
            print(f'  R export: {len(df)} records')

    elif DATA_SOURCE == 'GCP_DOWNLOAD':
        try:
            df = sig.load_motrpac_da(tissue, RAW_DIR, timepoint=TIMEPOINT)
            print(f'  GCP: {len(df)} records')
        except Exception as e:
            print(f'  GCP download failed ({e}); falling back to DEMO')
            df = make_demo_de(tissue)

    else:
        raise ValueError(f'Unknown DATA_SOURCE: {DATA_SOURCE}')

    # Basic stats
    n_sig = (df['adj_pvalue'] < FDR_CUTOFF).sum() if 'adj_pvalue' in df.columns else len(df)
    print(f'  Significant at FDR<{FDR_CUTOFF}: {n_sig}/{len(df)}')
    print(f'  logFC range: [{df["logFC"].min():.2f}, {df["logFC"].max():.2f}]')

    de_tables[tissue] = df

print('\nDE tables loaded for all tissues.')

In [ ]:
# ── Volcano plots — quick visual sanity check ─────────────────────────────────

fig, axes = plt.subplots(1, len(TISSUES), figsize=(5 * len(TISSUES), 4))
if len(TISSUES) == 1:
    axes = [axes]

for ax, tissue in zip(axes, TISSUES):
    df = de_tables[tissue]
    pval_col = 'adj_pvalue' if 'adj_pvalue' in df.columns else 'pvalue'
    neg_log_p = -np.log10(df[pval_col].clip(1e-300))
    sig_mask  = df[pval_col] < FDR_CUTOFF

    ax.scatter(df.loc[~sig_mask, 'logFC'], neg_log_p[~sig_mask],
               alpha=0.3, s=4, color='grey', rasterized=True)
    ax.scatter(df.loc[sig_mask & (df['logFC'] > 0), 'logFC'],
               neg_log_p[sig_mask & (df['logFC'] > 0)],
               alpha=0.5, s=6, color='firebrick', rasterized=True, label='up')
    ax.scatter(df.loc[sig_mask & (df['logFC'] < 0), 'logFC'],
               neg_log_p[sig_mask & (df['logFC'] < 0)],
               alpha=0.5, s=6, color='steelblue', rasterized=True, label='down')

    ax.axhline(-np.log10(FDR_CUTOFF), color='black', lw=0.8, ls='--')
    ax.set_title(sig.TISSUES[tissue]['display'], fontsize=10)
    ax.set_xlabel('log2 FC')
    ax.set_ylabel('-log10 adj.p')
    ax.legend(fontsize=8)

plt.suptitle(f'MoTrPAC — {TIMEPOINT} trained vs control', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/01_volcano_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved volcano plots.')

In [ ]:
# ── Load HCOP rat→human ortholog table ───────────────────────────────────────

print('Loading HCOP ortholog table …')
hcop = sig.load_hcop(EXTERNAL_DIR)
print(f'HCOP: {len(hcop):,} high-confidence rat→human ortholog pairs')
hcop.head(3)

In [ ]:
# ── Extract up/down gene sets per tissue ──────────────────────────────────────

gene_sets = {}   # {tissue: {'up': [...], 'down': [...]}}
all_records = []

for tissue in TISSUES:
    print(f'\n=== {tissue} ===')
    df = de_tables[tissue]

    gs = sig.extract_signatures(
        df, hcop,
        top_n=TOP_N,
        fdr_cutoff=FDR_CUTOFF,
        logfc_cutoff=LOGFC_CUTOFF,
    )
    gene_sets[tissue] = gs

    print(f'  Up genes:   {len(gs["up"])}')
    print(f'  Down genes: {len(gs["down"])}')
    print(f'  Up top-10: {gs["up"][:10]}')

    # Save gene lists
    sig.save_gene_set(gs['up'],   PROCESSED_DIR / f'motrpac_{tissue}_{TIMEPOINT}_up.txt')
    sig.save_gene_set(gs['down'], PROCESSED_DIR / f'motrpac_{tissue}_{TIMEPOINT}_down.txt')

    # Collect records for combined parquet
    for direction in ('up', 'down'):
        for gene in gs[direction]:
            all_records.append({'tissue': tissue, 'timepoint': TIMEPOINT,
                                 'direction': direction, 'human_symbol': gene})

    # Sanity check: are expected exercise genes present?
    result = sig.sanity_check_signature(gs['up'], tissue)
    print(f'  Sanity genes found: {result["sanity_genes_found"]}')

pd.DataFrame(all_records).to_parquet(PROCESSED_DIR / 'motrpac_signatures.parquet', index=False)
print('\nAll gene sets saved.')

In [ ]:
# ── Enrichr sanity check ──────────────────────────────────────────────────────
# Top-up genes per tissue should enrich for exercise-relevant Hallmark terms.
# Expected hits: OXIDATIVE_PHOSPHORYLATION, FATTY_ACID_METABOLISM, MYOGENESIS.

HALLMARK_DB = 'MSigDB_Hallmark_2020'
enrichr_results = {}

for tissue in TISSUES:
    up_genes = gene_sets[tissue]['up']
    print(f'Running Enrichr for {tissue} ({len(up_genes)} genes) …')
    try:
        enr = gp.enrichr(
            gene_list=up_genes,
            gene_sets=[HALLMARK_DB, 'KEGG_2021_Human'],
            outdir=None,
            verbose=False,
        )
        res = enr.results
        enrichr_results[tissue] = res
        top5 = res.nsmallest(5, 'Adjusted P-value')[['Term', 'Adjusted P-value', 'Overlap']]
        print(top5.to_string(index=False))
    except Exception as e:
        print(f'  Enrichr failed: {e} — check internet connection')
        enrichr_results[tissue] = pd.DataFrame()

print('\nEnrichr sanity check complete.')

In [ ]:
# ── Summary figure — gene set sizes and top Enrichr terms ────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: gene set sizes
ax = axes[0]
bar_data = []
for tissue in TISSUES:
    bar_data.append({'tissue': tissue, 'direction': 'up',   'n': len(gene_sets[tissue]['up'])})
    bar_data.append({'tissue': tissue, 'direction': 'down', 'n': len(gene_sets[tissue]['down'])})
bar_df = pd.DataFrame(bar_data)
sns.barplot(data=bar_df, x='tissue', y='n', hue='direction',
            palette={'up': 'firebrick', 'down': 'steelblue'}, ax=ax)
ax.set_title('Gene set sizes after ortholog mapping')
ax.set_xlabel('')
ax.set_ylabel('Number of human genes')
ax.axhline(TOP_N, ls='--', color='grey', lw=0.8, label=f'Target N={TOP_N}')
ax.legend()

# Panel B: Enrichr bubble plot for gastrocnemius (primary tissue)
ax = axes[1]
primary = TISSUES[0]
if not enrichr_results.get(primary, pd.DataFrame()).empty:
    top10 = enrichr_results[primary].nsmallest(10, 'Adjusted P-value').copy()
    top10['-log10(adj.p)'] = -np.log10(top10['Adjusted P-value'].clip(1e-20))
    top10['Term_short'] = top10['Term'].str.replace('HALLMARK_', '').str[:40]
    sns.barplot(data=top10, y='Term_short', x='-log10(adj.p)', orient='h',
                color='steelblue', ax=ax)
    ax.set_title(f'Enrichr top terms — {sig.TISSUES[primary]["display"]}')
    ax.set_xlabel('-log10 adj. p-value')
    ax.set_ylabel('')
else:
    ax.text(0.5, 0.5, 'Enrichr results unavailable', ha='center', va='center',
            transform=ax.transAxes)
    ax.set_title('Enrichr results')

plt.tight_layout()
plt.savefig('../results/figures/01_signature_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved signature summary figure.')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────

print('=== Notebook 01 complete ===')
print(f'Tissues processed: {TISSUES}')
print(f'Timepoint: {TIMEPOINT}')
print()
for tissue in TISSUES:
    gs = gene_sets[tissue]
    print(f'{tissue}:  {len(gs["up"])} up,  {len(gs["down"])} down genes')
print()
print('Files written to data/processed/')
for f in sorted(PROCESSED_DIR.glob('motrpac_*.txt')):
    print(f'  {f.name}')
print(f'  motrpac_signatures.parquet')
print()
print('→ Next: run notebook 02_lincs_query.ipynb')